In [5]:
# Ejercicio 5.3 - Titanic (Kaggle) con Keras (MLP)
#Pipeline für einen Kaggle

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1) Cargar CSV
df = pd.read_csv("/kaggle/input/titanic-dataset/Titanic-Dataset.csv")

print(df.head())
print(df.columns)


# ['PassengerId','Survived','Pclass','Name','Sex','Age',
#  'SibSp','Parch','Ticket','Fare', ... evtl. Cabin, Embarked]

# 2) Selección de características y objetivo
target_col = "Survived"
feature_cols = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]  # einfache, sinnvolle Features

data = df[feature_cols + [target_col]].copy()

# 3) Tratamiento de valores faltantes
# Age und ggf. Fare können NaNs haben -> mit median füllen
for col in ["Age", "Fare"]:
    if col in data.columns:
        data[col] = data[col].fillna(data[col].median())

# 4) Codificación de variables categóricas
# 'Sex' -> 0/1; wenn 'Embarked' oder andere Kategorische Features genutzt werden, get_dummies()
data["Sex"] = data["Sex"].map({"male": 0, "female": 1}).astype("int32")

# Optional: One-Hot-Encoding für Pclass (statt numerisch)
# data = pd.get_dummies(data, columns=["Pclass"], drop_first=True)

X = data.drop(columns=[target_col])
y = data[target_col].astype("int32")

print("\nX shape:", X.shape)
print("y value counts:\n", y.value_counts())

# 5) Train/Test-Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6) Escalado de características numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

input_dim = X_train_scaled.shape[1]

# 7) Definición del modelo (MLP)
model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),   # salida binaria: probabilidad de sobrevivir
])

model.summary()

# 8) Compilación
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 9) Entrenamiento
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# test
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc:.4f} | loss: {test_loss:.4f}")

# 11) Métricas
y_pred_prob = model.predict(X_test_scaled, verbose=0).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

#  predicción
sample = X_test.iloc[:5]
sample_scaled = scaler.transform(sample)
sample_pred_prob = model.predict(sample_scaled, verbose=0).ravel()
sample_pred = (sample_pred_prob >= 0.5).astype(int)

print("\nEjemplos de predicción (primeras 5 filas del test):")
for i in range(len(sample)):
    print(f"Fila {sample.index[i]} -> ProbSurv={sample_pred_prob[i]:.3f}, "
          f"Pred={sample_pred[i]}, Real={y_test.iloc[i]}")


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
In

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 769 (3.00 KB)

 Trainable params: 769 (3.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.7012 - loss: 0.6402 - val_accuracy: 0.7063 - val_loss: 0.6168
Epoch 2/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7557 - loss: 0.5634 - val_accuracy: 0.7552 - val_loss: 0.5551
Epoch 3/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7821 - loss: 0.5126 - val_accuracy: 0.7832 - val_loss: 0.5170
Epoch 4/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7838 - loss: 0.4807 - val_accuracy: 0.7902 - val_loss: 0.4941
Epoch 5/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7979 - loss: 0.4592 - val_accuracy: 0.7902 - val_loss: 0.4775
Epoch 6/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.7996 - loss: 0.4455 - val_accuracy: 0.7902 - val_loss: 0.4685
Epoch 7/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.8014 - loss: 0.4350 - val_accuracy: 0.7972 - val_loss: 0.4638
Epoch 8/40
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8067 - loss: 0.4283 - val_accuracy: 0.8112 - v